In [ ]:
pip install findspark

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [2]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [ ]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{osql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.


In [ ]:
# Paste the following modified code into your active PySpark shell session
import time
# --- Configuration (UPDATE THESE VALUES) ---
# Start the timer
start_time = time.time()
JDBC_URL = "jdbc:sqlserver://HYPE\\SQL_2022:1433;"
SOURCE_DB_NAME = "AdventureSales" 
TARGET_TABLE = "Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint"
USERNAME = "sa"
PASSWORD = "D@1ry$2014"
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" 

# Connection Properties for the READ operation (no change needed here)
read_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
}

# Connection Properties for the WRITE operation
# We rely on the default OVERWRITE behavior (DROP and CREATE TABLE)
write_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
    # Remove "truncate": "true" here
    # --- ADD THIS LINE FOR BULK INSERT SPEED ---
    "bulkCopy": "true" 
}


# --- 1. Define the Source SQL Query (No change) ---
source_sql_query = """
(SELECT 
    *
FROM sttwk1b 
) as custom_query_alias
"""

# --- 2. Load data from SQL Server into a PySpark DataFrame ---
source_df = spark.read.jdbc(
    url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
    table=source_sql_query,
    properties=read_properties
)

print(f"Loaded {source_df.count()} records from source SQL query into PySpark DataFrame.")
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds") 

# --- 3. Overwrite the Target Table (DROP and CREATE TABLE approach) ---
# Start the timer
start_time = time.time()
optimized_df = source_df.repartition(16)
try:
    # Use the optimized DataFrame
    optimized_df.write \
        .mode("overwrite") \
        .jdbc(
            url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
            table=TARGET_TABLE, 
            properties=write_properties # Make sure you use the bulkCopy properties
        )
    print(f"Successfully wrote data with increased parallelism.")
except Exception as e:
    print(f"An error occurred during write operation: {e}")
    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds") 